# Choosing fewer b-values: can we keep the accuracy of *your* diffusion protocol with fewer measurements?

> **DRAFT proposal for discussion (Paddy has the last word).** It keeps the ED_MRI/TADRED core of the
> original draft, but starts from the b-values of the participants' own sequence and leaves the
> simulation, fitting and comparison for them to write.

In the sequence session you designed a diffusion acquisition and ended up with a **list of b-values**.
Every extra b-value costs scan time, which matters in the placenta. So:

**Which smaller subset of *your* b-values estimates the diffusion parameters almost as well as all of them?**

We use [TADRED](https://github.com/PaddySlator/tadred) (through ED_MRI) to choose the subset. It is a
machine-learning method that looks at simulated data for *all* of your b-values and keeps only the
measurements the parameter estimates really need. The optimiser is a black box here: your work is to set
the problem up, run it, and **check whether the subset it picks is actually good**.

## 1. Your starting point: the b-values of your sequence

In [ ]:
import subprocess
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# The b-values (s/mm2) your sequence produces, one per acquisition, from the sequence session.
# TODO: point this at your own file (or paste your list). One b-value can appear several times.
BVALS_FILE = None
b_all = np.loadtxt(BVALS_FILE) if BVALS_FILE else None

if b_all is None:                       # stand-in so that the notebook runs before you have yours
    print("No b-values file given: using an example protocol. Replace it with your own!")
    b_all = np.array([0, 0, 50, 100, 150, 200, 300, 400, 500, 600, 700, 800, 1000, 1200, 1500, 2000], float)
print(len(b_all), "b-values:", b_all)

## 2. Simulate signals for your protocol

We need synthetic voxels with *known* parameters, measured at *your* b-values and with noise.

Start with **ordinary diffusion** (one parameter, the ADC, `D`): `S(b) = exp(-b D)`.
Later (extension) you can move to IVIM, which adds a fast "perfusion" compartment.

**Task:** write `adc_signal(b, D)`, draw many `D` values from a plausible range, and add noise.
Noise and fitting are treated properly in the Day 3 talks, so a simple model is enough for now:
Rician noise is `sqrt((S + n1)**2 + n2**2)` with `n1, n2 ~ N(0, 1/SNR)`.

In [ ]:
def adc_signal(b, D):
    return np.exp(-np.outer(D, b))

def add_noise(signal, snr):
    s = 1 / snr
    return np.sqrt((signal + np.random.normal(0, s, signal.shape))**2 + np.random.normal(0, s, signal.shape)**2)

n_voxels, snr = 1000, 30
D_range = (0.0002, 0.003)
D_true = np.random.uniform(*D_range, n_voxels)

## 3. Baseline: how well do *all* your b-values do?

**Task:** fit `D` voxel by voxel to the noisy signals (non-linear least squares, `scipy.optimize.least_squares`
or `curve_fit`, or the fitting code you already have) and measure the error.
A good summary is the RMSE (or the correlation) between true and fitted `D`.
This number is the one every smaller protocol has to be compared with.

In [ ]:
from scipy.optimize import least_squares
def fit_adc(b, signals):
    return np.array([least_squares(lambda p: np.exp(-b*p[0]) - s, [1e-3], bounds=([0],[0.01])).x[0] for s in signals])
def rmse(true, est):
    return float(np.sqrt(np.mean((np.asarray(true) - np.asarray(est))**2)))
signals_all = add_noise(adc_signal(b_all, D_true), snr)
D_fit_all = fit_adc(b_all, signals_all)
print("RMSE, all b-values:", rmse(D_true, D_fit_all))

## 4. Let TADRED choose a subset of your b-values

TADRED needs three things, which the next cell writes to disk: the signals at **all** your b-values
(`input`), the true parameters (`target`), and your b-values (`superdesign`: the candidate measurements).
It then keeps a fraction `protocol_size` of them and trains a neural network that estimates the parameters
from just those.

**Task:** pick how many b-values you want to keep (`n_keep`) and run it.

In [ ]:
from edmri.paths import CONFIG_PATH, RESULTS_PATH
np.savetxt("my_bvals.txt", b_all); np.save("my_signals.npy", signals_all); np.save("my_targets.npy", D_true[:, None])
n_keep = 4
protocol_size = n_keep / len(b_all)
subprocess.run(f"PYTHONUNBUFFERED=1 edmri-optimise --force --example ADC --config {CONFIG_PATH}/optimise_default.yaml data.input=my_signals.npy data.target=my_targets.npy data.superdesign=my_bvals.txt optimisation.protocol_size={protocol_size}", shell=True, check=True)

## 5. Is the chosen subset any good?

TADRED saved the b-values it kept (`optimised_protocol.txt`) and the trained network.

**Tasks**

1. Load the selected b-values. Which ones did it pick? Plot them on top of your full list. Does it make sense?
2. Simulate **fresh** noisy voxels at the selected b-values, and estimate `D` in two ways:
   least squares on the subset, and the trained network (`edmri-apply`, see the original cell below).
3. Compare with **two reference protocols of the same size**: a random subset and an evenly-spaced subset
   of your b-values. If TADRED does not beat those, the "optimisation" is not buying anything.

In [ ]:
out = Path(RESULTS_PATH, "optimisation/examples/ADC/tadred_results")
b_sel = np.loadtxt(out / "optimised_protocol.txt")
print("selected b-values:", b_sel)
D_new = np.random.uniform(*D_range, 2000)
rng = np.random.default_rng(0)
subsets = {"TADRED": b_sel, "random": np.sort(rng.choice(b_all, len(b_sel), replace=False)),
           "even": b_all[np.linspace(0, len(b_all)-1, len(b_sel)).round().astype(int)]}
for name, b in subsets.items():
    print(name, rmse(D_new, fit_adc(b, add_noise(adc_signal(b, D_new), snr))))
np.save("signals_sel.npy", add_noise(adc_signal(b_sel, D_new), snr))
subprocess.run(["edmri-apply", "--force", "--example", "ADC", "--config", f"{CONFIG_PATH}/apply_default.yaml",
                "data.input=signals_sel.npy", f"task_network={out}/trained_task_network.pt"], check=True)
D_net = np.load(Path(RESULTS_PATH, "application/examples/ADC/predictions.npy"))
print("network", rmse(D_new, D_net.ravel()))

## 6. Go further (pick what interests your team)

1. **Accuracy against number of measurements.** Repeat sections 4-5 for several `n_keep` and plot the error
   against it, for TADRED, random and evenly-spaced subsets. Where is the knee?
2. **Move from ADC to IVIM** (`S = f exp(-b D*) + (1-f) exp(-b D)`, three parameters; see the Additional-tasks list
   of the original draft). Does the answer change? Which low b-values does it now keep?
3. **Noise.** Change the SNR. Does the best subset change?
4. **Your protocol itself.** Do the selected b-values suggest a change to the sequence you designed? What would you
   drop, or repeat? (This is the loop back to the sequence stage of the pipeline.)
5. **Optimisation settings.** Copy `optimise_default.yaml`, change the network size or the number of reduction stages.

**Overall challenge:** the smallest subset of *your* b-values that keeps parameter-estimation accuracy.